In [1]:
from jinja2 import Template
import pandas as pd
import os
import yaml
import sys
import json
import random
import hashlib
from itertools import product
import torch as t
from transformers import AutoTokenizer, AutoModelForCausalLM
from pydantic import BaseModel
from typing import Union
import outlines
from outlines import Generator
from openai import OpenAI
from tqdm import tqdm
sys.path.append("../")
from src.utils_v0 import list_to_str, openai_api_call
device = "cpu"

In [2]:
def write_to_json(file, file_path):
    with open(file_path, 'w') as f:
        json.dump(file, f)
        
def read_json(file_path):
    with open(file_path, "r") as f:
        file = json.load(f)
    return file

def expand_dict_combinations(data):
    keys = list(data.keys())
    values = [data[k] for k in keys]
    
    combos = []
    for prod in product(*values):
        combos.append(dict(zip(keys, prod)))
    return combos

def generate_hash(prompt):
    hash_object = hashlib.sha256()
    hash_object.update(prompt.encode("utf-8"))
    return hash_object.hexdigest()

In [3]:
class SyntheticSJT(BaseModel):
    question: str
    honesty_humility_option: str
    emotionality_option: str
    extraversion_option: str
    agreeableness_option: str
    conscientiousness_option: str
    openness_option: str

class TraitBleedEval(BaseModel):
    score: int
    analysis: str
    suggested_correction: Union[str, None]

class HexacoTrait(BaseModel):
    honesty_humility: TraitBleedEval
    emotionality: TraitBleedEval
    extraversion: TraitBleedEval
    agreeableness: TraitBleedEval
    conscientiousness: TraitBleedEval
    openness: TraitBleedEval
    
class SjtTraitBleedEval(BaseModel):
    scenario_summary: str
    trait_evaluations: HexacoTrait
    corrected_sjt: SyntheticSJT
    overall_notes: str

In [4]:
with open('../configs/synthetic_sjt_seeds.yaml', 'r') as file:
    synthetic_sjt_seeds = yaml.safe_load(file)

handmade_sjt_template_df = pd.read_csv("sjt_data/sjt_jinja_template_v2.csv")

In [5]:
all_seed_combos = expand_dict_combinations(synthetic_sjt_seeds)

In [6]:
len(all_seed_combos)

6531840

In [7]:
random.seed(42)
n = 60
sampled_seed_combos =  random.sample(all_seed_combos, n)

In [8]:
SJT_GENERATION_TEMPLATE_STR = """You are creating a new law enforcement Situational Judgment Test (SJT) scenario by modifying an existing scenario with new attribute values. Follow the template below to generate a realistic, professionally appropriate scenario that maintains the core decision-making structure while incorporating the specified attributes.

**Base Scenario to Modify:**
{{base_scenario}}


**Instructions**
* Keep the same personality trait dimensions being tested (Honesty-Humility, Emotionality, Extraversion, Agreeableness, Conscientiousness, Openness to Experience)
* Preserve the professional law enforcement context

**Attribute description**
* **Urgency Level:** Adjust time pressure and decision timeline accordingly
   * Low: Allow deliberation time, non-critical timing
   * Medium: Some time pressure, manageable deadlines
   * High: Immediate decisions required, critical timing
* **Threat Level:** Scale physical danger and safety concerns
   * Low: Administrative issues, minor policy matters
   * Medium: Potential for injury, safety protocols needed
   * High: Life-threatening situations, lethal force considerations
* **Ambiguity Level:** Modify clarity of protocols and guidance
   * Clear: Obvious procedures, straightforward application
   * Moderate: Some judgment required, minor gray areas
   * High: Conflicting guidance, novel situations, ethical dilemmas
* **Individuals Involved:** Adjust scenario complexity
   * Simple: Officer making individual decision
   * Moderate: 2-3 parties with different perspectives
   * Complex: Multiple stakeholders, witnesses, supervisors
* **Authority Relationships:** Frame interactions appropriately
   * Peer Level: Fellow officers, partners, colleagues
   * Subordinate: Supervisors, training officers, senior personnel
   * Authority: Suspects, witnesses, civilians, subordinates
* **Ethical Considerations:** Incorporate specified ethical tension
* **Situation Type:** Adapt setting and context to match type
* **Time of Day:** Include time context naturally in scenario
* **Demographics:** Integrate subject's race, gender, and age naturally without stereotyping if applicable.

**New Attribute Values:**
* **Urgency Level:** {{urgency_level}}
* **Threat Level:** {{threat_level}}
* **Ambiguity Level:** {{ambiguity_level}}
* **Individuals Involved:** {{individuals_involved}}
* **Authority Relationships:** {{authority_relationships}}
* **Ethical Considerations:** {{ethical_considerations}}
* **Situation Type:** {{situation_type}}
* **Time of Day:** {{time_of_day}}
* **Subject Race:** {{race}}
* **Subject Gender:** {{gender}}
* **Subject Age:** {{age}}


**Response Options Requirements**
Generate six response options (1-6) that:
* Clearly differentiate the six personality dimensions
* Remain professionally appropriate and realistic
* Reflect how each personality type would approach the modified scenario
* Maintain consistent quality and plausibility across all options
* The responses are qualitative descriptions of actions and do not contain specific speech suggestions.
* Integrate all specified attributes naturally into the scenario without explicitly naming them
* Do not use direct labels like "high-priority," "mental health crisis," "high threat," etc. in the responses.
* Make attribute levels apparent through context, actions, and circumstances rather than descriptive terms
* Let the urgency, threat level, and situation type emerge from the scenario details rather than being stated outright
* Ensure scenario is realistic and could occur in actual law enforcement
* The scenario should be 2-4 sentences
* Include specific, actionable decisions rather than vague choices
* Verify that all six personality responses are distinct and characteristic

**Output Format**
Provide the complete SJT scenario followed by the six response options as a JSON object with the following schema:

{
  'question': '<string>',
  'honesty_humility_option': '<string>',
  'emotionality_option': '<string>',
  'extraversion_option': '<string>',
  'agreeableness_option': '<string>',
  'conscientiousness_option': '<string>',
  'openness_option': '<string>'
}

Do not include any extra text, explanation, or formatting outside of the JSON object.
"""

SJT_TRAIT_BLEED_EVALUATION_TEMPLATE_STR = """ You are an expert SJT evaluator and corrector specializing in HEXACO-aligned scenarios.  You are given a situational judgment test (SJT) scenario with six answer options, each intended  to correspond to one HEXACO trait: Honesty-Humility, Emotionality, Extraversion, Agreeableness,  Conscientiousness, and Openness.

Your tasks:
1. **Trait Fit Evaluation**  
   - For each option, evaluate how strongly it aligns with its intended trait definition.  
   - Use a 1–5 scale:  
     5 = Very strong, clean representation, no leakage  
     4 = Strong but with minor overlap  
     3 = Moderate, noticeable blending  
     2 = Weak, trait unclear or diluted  
     1 = Poor, option does not represent the trait well  

2. **Separation Analysis**  
   - Highlight where options overlap or bleed into each other (e.g., Extraversion vs. Agreeableness).  
   - Explain why the overlap occurs.  

3. **Correction Suggestions**  
   - For any option rated below 5, propose a corrected rewrite that emphasizes the target trait more cleanly.  
   - Ensure each rewrite minimizes overlap with other traits.  
   - Ensure each rewrite include specific, actionable decisions rather than vague choices.

4. **Final Corrected SJT Object**  
   - Output an object with the exact same structure as the input SJT dictionary.  
   - Each option should contain the corrected version if a rewrite was needed, or the unchanged original if not.  

5. **Output Format**  
   Return results in structured JSON with this format:

{
  "scenario_summary": "<1-2 sentence summary of scenario>",
  "trait_evaluations": {
    "honesty_humility": {
      "score": <1-5>,
      "analysis": "<why it fits/doesn't fit>",
      "suggested_correction": "<if needed, otherwise null>"
    },
    "emotionality": {
      "score": <1-5>,
      "analysis": "<why it fits/doesn't fit>",
      "suggested_correction": "<if needed, otherwise null>"
    },
    "extraversion": {
      "score": <1-5>,
      "analysis": "<why it fits/doesn't fit>",
      "suggested_correction": "<if needed, otherwise null>"
    },
    "agreeableness": {
      "score": <1-5>,
      "analysis": "<why it fits/doesn't fit>",
      "suggested_correction": "<if needed, otherwise null>"
    },
    "conscientiousness": {
      "score": <1-5>,
      "analysis": "<why it fits/doesn't fit>",
      "suggested_correction": "<if needed, otherwise null>"
    },
    "openness": {
      "score": <1-5>,
      "analysis": "<why it fits/doesn't fit>",
      "suggested_correction": "<if needed, otherwise null>"
    }
  },
  "corrected_sjt": {
    "question": "<original or unchanged question>",
    "honesty_humility_option": "<corrected or unchanged option>",
    "emotionality_option": "<corrected or unchanged option>",
    "extraversion_option": "<corrected or unchanged option>",
    "agreeableness_option": "<corrected or unchanged option>",
    "conscientiousness_option": "<corrected or unchanged option>",
    "openness_option": "<corrected or unchanged option>"
  },
  "overall_notes": "<high-level summary of trait separation quality>"
}

---

### SJT Input
Here is the SJT you must evaluate:

{
  "question": {{ question }},
  "honesty_humility_option": {{ honesty_humility_option }},
  "emotionality_option": {{ emotionality_option }},
  "extraversion_option": {{ extraversion_option }},
  "agreeableness_option": {{ agreeableness_option }},
  "conscientiousness_option": {{ conscientiousness_option }},
  "openness_option": {{ openness_option }}
}

"""

In [9]:
sjt_example_template_str = """

Question: {{ question }}
Options: 

{{ answer_options}}

"""

In [10]:
sjt_example_template = Template(sjt_example_template_str)
SJT_GENERATION_TEMPLATE = Template(SJT_GENERATION_TEMPLATE_STR)
SJT_TRAIT_BLEED_EVALUATION_TEMPLATE = Template(SJT_TRAIT_BLEED_EVALUATION_TEMPLATE_STR)

In [11]:
option_cols = ['Option 1', 'Option 2', 'Option 3', 'Option 4', 'Option 5','Option 6']

In [75]:
synthetic_generated_sjt_list = []
handmade_sjt_sample = handmade_sjt_template_df.sample(3)
for index, row in tqdm(handmade_sjt_sample.iterrows(), desc = "base_scenario", position=0):
    question = row['Question']
    answer_options = list_to_str(row[option_cols])
    
    base_scenario = sjt_example_template.render(question = question, answer_options=answer_options)
    sampled_seeds = random.sample(sampled_seed_combos,4)
    for seed_dict in tqdm(sampled_seeds, desc = "seeds", position=1):
        generated_sjt_dict = {}
        # generated_sjt_dict['base_scenario'] = base_scenario
        seed_copy = seed_dict.copy()
        seed_copy['base_scenario'] = base_scenario
        sjt_generation_prompt = SJT_GENERATION_TEMPLATE.render(seed_copy)
        generated_sjt_dict['config'] = seed_copy
        generated_sjt_dict['sjt_generation_prompt'] = sjt_generation_prompt
        openai_sjt_response_v1 = openai_api_call(prompt=sjt_generation_prompt, response_format=SyntheticSJT ,model="gpt-4.1")
        original_sjt_response_dict = openai_sjt_response_v1.model_dump()
        
        # Evaluating the created SJT for trait bleed and correcting it if there is any correction needed
        sjt_trait_bleed_evaluation_prompt = SJT_TRAIT_BLEED_EVALUATION_TEMPLATE.render(original_sjt_response_dict)
        openai_sjt_response_v2 = openai_api_call(prompt=sjt_trait_bleed_evaluation_prompt, response_format=SjtTraitBleedEval ,model="gpt-4.1")
        
        corrected_sjt_response_dict = openai_sjt_response_v2.model_dump()
        
        generated_sjt_dict['hash_id'] = generate_hash(json.dumps(corrected_sjt_response_dict))
        generated_sjt_dict['original_sjt'] = original_sjt_response_dict
        generated_sjt_dict['trait_bleed_evaluation'] = corrected_sjt_response_dict
        generated_sjt_dict['corrected_sjt'] = corrected_sjt_response_dict['corrected_sjt']
        synthetic_generated_sjt_list.append(generated_sjt_dict)
        break
    break

seeds:   0%|          | 0/4 [00:40<?, ?it/s]
base_scenario: 0it [00:40, ?it/s]


In [13]:
generated_sjt_list = read_json("sjt_data/synthetic_generated_sjt_list_v8_temp_1point5.json")

sjt_keys = ['question', 'honesty_humility_option', 'emotionality_option', 'extraversion_option', 'agreeableness_option', 'conscientiousness_option', 'openness_option']

correct_sjt_list = []
for sjt in tqdm(generated_sjt_list, desc = "generated_sjts"):
    corrected_sjt_dict = {}
    sjt_copy = sjt.copy()
    filtered_sjt = {k: sjt_copy[k] for k in sjt_keys}
    
    sjt_trait_bleed_evaluation_prompt = SJT_TRAIT_BLEED_EVALUATION_TEMPLATE.render(filtered_sjt)
    
    openai_sjt_response_v2 = openai_api_call(prompt=sjt_trait_bleed_evaluation_prompt, response_format=SjtTraitBleedEval ,model="gpt-4.1",temperature=1.5,
                                              top_p=0.95)
    
    corrected_sjt_response_dict = openai_sjt_response_v2.model_dump()
    
    corrected_sjt_dict.update(sjt_copy)
    corrected_sjt_dict['hash_id'] = generate_hash(json.dumps(corrected_sjt_response_dict))
    corrected_sjt_dict['trait_bleed_evaluation'] = corrected_sjt_response_dict
    corrected_sjt_dict['corrected_sjt'] = corrected_sjt_response_dict['corrected_sjt']
    correct_sjt_list.append(corrected_sjt_dict)

write_to_json(correct_sjt_list,"sjt_data/synthetic_generated_sjt_list_v8.1_temp_1point5.json")

generated_sjts: 100%|█████████████████████████████████████████████████████████████████| 56/56 [30:12<00:00, 32.37s/it]


{'question': 'During a night patrol traffic stop, you and your training officer pull over a middle-aged driver of unknown gender and Asian descent. As you approach, you detect a strong smell similar to cannabis, and you note heightened nervousness in the car’s occupants. The driver denies consent to search, but your training officer discreetly suggests that presenting the situation as riskier than it appears might secure legal justification for a warrantless search. Given limited backup nearby, conflicting policies on odor searches, and loyalty tension with your supervising partner, you must choose how to proceed.',
 'honesty_humility_option': 'You state that accuracy in your observations is essential and refuse to embellish facts or circumstances for the report, knowing honesty is more important than pleasing your training officer. You accept personal responsibility even if it means disappointing or disagreeing with your superior.',
 'emotionality_option': 'You acknowledge feeling anx

In [ ]:
write_to_json(synthetic_generated_sjt_list,"sjt_data/synthetic_generated_sjt_list_v3.json")

In [3]:
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

In [4]:
def qa_to_string(entry: dict) -> str:
    parts = []
    # Add question first
    parts.append(f"Question: {entry.get('question', '')}")
    
    # Add each trait option
    for key, value in entry.items():
        if key != "question":
            parts.append(f"{key.replace('_option', '').capitalize()}: {value}")
    
    # Join all parts into one string
    return "\n".join(parts)

def embed_sjt(sjt):
    
    corrected_sjt = sjt['corrected_sjt']
    sjt_string = qa_to_string(corrected_sjt)
    
    return embed_model.encode([sjt_string], convert_to_numpy=False)[0]

In [5]:
embed_model = SentenceTransformer("Qwen/Qwen3-Embedding-0.6B")

In [6]:
%time embed_sjt(complete_sjt_list[0]).cpu()

NameError: name 'complete_sjt_list' is not defined

{'question': "During a routine afternoon, you are finishing up a paperwork review at the station with a colleague. Another department requests your assistance via radio regarding an ongoing investigation; specifically, they ask for you to check surveillance logs for a case involving a female Pacific Islander subject of unknown age. The request is straightforward, and protocols clearly outline inter-agency cooperation steps. However, you and your colleague have both been cautious in the past about how much information is shared outside your agency. You must decide whether to share the information fully and transparently, balancing the need for open cooperation against potential agency privacy considerations, knowing your colleague's opinion might differ from yours.",
 'honesty_humility_option': 'You provide complete and accurate information to the requesting department exactly as outlined in protocol, explicitly prioritizing transparency and fairness in information-sharing without conce

In [7]:
data_dir = "sjt_data"
complete_sjt_list = []
for dir_name in os.listdir(data_dir):
    if "synthetic_generate_sjt_1k_temp1point5" in dir_name:
        print(dir_name)
        for filename in tqdm(os.listdir(os.path.join(data_dir, dir_name)), desc = "filename"):
            template_no = filename.split("_")[-1].split(".")[0]
            file = read_json(os.path.join(data_dir, dir_name, filename))
            sjt_list = []
            for sjt in tqdm(file, desc = "sjts"):
                sjt_embedding = embed_sjt(sjt)
                sjt_list.append(sjt | {"template_no":template_no,
                              "sjt_embedding": sjt_embedding})

            complete_sjt_list.extend(sjt_list)

synthetic_generate_sjt_1k_temp1point5_v3


filename:  25%|█████████████████▌                                                    | 5/20 [02:58<08:54, 35.65s/it]


KeyboardInterrupt: 

In [6]:
from datasets import Dataset

sjt_dataset = Dataset.from_list(synthetic_generated_sjt_list)


# sjt_dataset.to_parquet("sjt_data/sjt_hf_dataset_sample_v2.parquet")

In [22]:
from huggingface_hub import login
login("")

In [8]:
sjt_dataset.push_to_hub("thoughtworks/psychometric_SJTs")

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

README.md:   0%|          | 0.00/24.0 [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/datasets/thoughtworks/psychometric_SJTs/commit/6319db543b2f4e9f7dfd0ad1f96f4b80a2dbb584', commit_message='Upload dataset', commit_description='', oid='6319db543b2f4e9f7dfd0ad1f96f4b80a2dbb584', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/thoughtworks/psychometric_SJTs', endpoint='https://huggingface.co', repo_type='dataset', repo_id='thoughtworks/psychometric_SJTs'), pr_revision=None, pr_num=None)